This notebooks works through all harmonized ST datasets, subsetting to only cells found within the defined CCF bounding box

Operation(s):

1. Wrap z coordinates around midline (adds .obsm['coords_ccf_mm_wrap])
2. Add voxel coordinate systems (add .obsm['coords_voxels_25um'], .obsm['coords_voxels_25um_bounded'])
3. Subset ST datasets to only cells within the Hypothalamic Bounding Box
4. Add a boolean annotation for whether a spatial cell resides in the HY or just in bounding box peripheral regions (adds .obs['IsHY'])

Output(s):

1. DATADIR/7_spatial/bounded/SP*-**.h5ad

Requirement(s):

1. DATADIR/7_spatial/harmonized/SP*-**.h5ad
2. DATADIR/7_spatial/bounding_box_ccf_mm.json

In [1]:
import warnings
import json

from pathlib import Path

import anndata as ad
import numpy as np

from dotenv import dotenv_values

from hm3d_utils.spatial import BoundingBox3D

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial"
(OUTDIR / "bounded").mkdir(exist_ok=True, parents=True) 

### Loading the 3D Bounding Box

In [3]:
bbox_fp = OUTDIR / 'bounding_box_ccf_mm.json'
bbox_ccf = BoundingBox3D.load(bbox_fp)

bbox_fp = OUTDIR / 'bounding_box_voxels_25um.json'
bbox_voxel = BoundingBox3D.load(bbox_fp)

bbox_ccf, bbox_voxel

(Bounding Box Ranges (CCFv3 mm):
  x: 4.375<->8.55
  y: 3.95<->7.525
  z: 3.05<->5.7,
 Bounding Box Ranges (25um Voxels):
  x: 175.0<->342.0
  y: 158.0<->301.0
  z: 122.0<->228.0)

In [4]:
z_midline = bbox_ccf.max[2]
z_midline

np.float64(5.7)

### Get list of all subregions of HY

In [5]:
infile = OUTDIR / 'structure_tree_processed.json'
with open(infile, 'r') as fp:
    structure_tree = json.load(fp)
    

subregions = structure_tree['1097']['subregions'] # where 1097 is ID of hypothalamus

### Run per dataset

In [6]:
harmonized_dir = OUTDIR / 'harmonized'
for dataset_fp in sorted(harmonized_dir.glob('SP*-*.h5ad')):

    adata = ad.read_h5ad(dataset_fp)
    
    # Wrap z coordinate
    z_coord = adata.obsm['coords_ccf_mm']['z'].values
    z_coord_wrapped = z_midline - np.abs(z_midline - z_coord)
    adata.obsm['coords_ccf_mm_wrap'] = adata.obsm['coords_ccf_mm'].copy()
    adata.obsm['coords_ccf_mm_wrap']['z'] = z_coord_wrapped
    
    # Create voxel coordinate systems
    adata.obsm['coords_voxels_25um'] = np.floor(adata.obsm['coords_ccf_mm_wrap'] * 1000 / 25).astype(int)
    adata.obsm['coords_voxels_25um_bounded'] = (adata.obsm['coords_voxels_25um'] - bbox_voxel.min).astype(int)
    
    # Identify valid obs
    inbounds_mask = bbox_voxel.isinbounds(adata.obsm['coords_voxels_25um'].values)
    inbounds_adata = adata[inbounds_mask].copy()
    
    # Add IsHY column
    inbounds_adata.obs['IsHY'] = inbounds_adata.obs['RegionID'].isin(subregions)
    
    # Save out subset dataset
    outfile = OUTDIR / 'bounded' / dataset_fp.name
    inbounds_adata.write_h5ad(outfile)
    
    print(f"Dataset {dataset_fp.name} was subset from {adata.n_obs} to {inbounds_adata.n_obs} obs of which {np.sum(inbounds_adata.obs['IsHY'])} are hypothalamic.")

Dataset SP2-01.h5ad was subset from 541968 to 510968 obs of which 130826 are hypothalamic.
Dataset SP2-01_no_qc.h5ad was subset from 545586 to 514319 obs of which 131387 are hypothalamic.
Dataset SP2-02.h5ad was subset from 562286 to 529710 obs of which 136357 are hypothalamic.
Dataset SP2-02_no_qc.h5ad was subset from 709698 to 671081 obs of which 169329 are hypothalamic.
Dataset SP2-03.h5ad was subset from 153117 to 147316 obs of which 75608 are hypothalamic.
Dataset SP2-03_no_qc.h5ad was subset from 600184 to 500570 obs of which 203552 are hypothalamic.
Dataset SP2-04.h5ad was subset from 225780 to 213319 obs of which 46588 are hypothalamic.
Dataset SP2-04_no_qc.h5ad was subset from 1784666 to 1692576 obs of which 435517 are hypothalamic.
